# Content-based recommender system
## UK online retail
### This notebook uses the *Online Retail.xlsx* dataset

Content-based recommendations systems recommend items that are similar to items previously purchased by other customers. The models used in content-based recommender systems identify common features across customer preferences based on the historical purchase data of the customer (e.g., what items did customers who made similar purchases also made).
<br>In this example, there are two objectives:
- Develop a system to recomend products to a customer, based on similar customers (customers that buy similar products).
- Develop a system to recommend products based on products that are usualy sold with a specific product.
<p/>This dataset is a transnational data set which contains all the transactions occurring between 01/12/2010 and 09/12/2011 for a UK-based and registered non-store online retail.The company mainly sells unique all-occasion gifts. Many customers of the company are wholesalers.</p>
<p>More information at https://archive.ics.uci.edu/ml/datasets/online+retail</p>
<p>Some parts adapted from https://github.com/yoonhwang/hands-on-data-science-for-marketing/blob/master/ch.6/python/ProductRecommendation.ipynb</p>

This is a simple example. Not all CRISP-DM phases are described in high detail.

(c) Nuno António, João Silveira, Hugo Silva 2023 - Rev. 1.21

Each row in the dataset represents a document line (invoice line). The dataset is composed of the following columns:
- **InvoiceNo**: Invoice number. Nominal, a 6-digit integral number uniquely assigned to each transaction. If this code starts with letter 'c', it indicates a cancellation.
- **StockCode**: Product (item) code. Nominal, a 5-digit integral number uniquely assigned to each distinct product.
- **Description**: Product (item) name. Nominal.
- **Quantity**: The quantities of each product (item) per transaction. Numeric.
- **InvoiceDate**: Invice Date and time. Numeric, the day and time when each transaction was generated.
- **UnitPrice**: Unit price. Numeric, Product price per unit in sterling.
- **CustomerID**: Customer number. Nominal, a 5-digit integral number uniquely assigned to each customer.
- **Country**: Country name. Nominal, the name of the country where each customer resides.

## Initial setup and modules/packages loading

In [1]:
# Packages
import os
import numpy as np
import pandas as pd
import re

In [2]:
# Distances functions
#from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import pairwise_distances

# For more info on how to implement in Python check https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise_distances.html#examples-using-sklearn-metrics-pairwise-distances and https://docs.scipy.org/doc/scipy/reference/spatial.distance.html

In [3]:
# Global definitions
baseFolder = os.getcwd()
exportsFolder = baseFolder + os.sep +'Exports' + os.sep

## Data understanding

In [4]:
# Load data 
ds = pd.read_excel('Online Retail.xlsx', engine='openpyxl')
ds.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [5]:
# Reformat variable types
cols = ['InvoiceNo','StockCode','Description','Country','CustomerID']
ds[cols] = ds[cols].astype(str)

In [6]:
# Summary statistics for all variables
ds.describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
InvoiceNo,541909,25900,573585,1114,NaN,NaN,NaN,NaN,NaN,NaN,NaN
StockCode,541909,4070,85123A,2313,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Description,541909,4224,WHITE HANGING HEART T-LIGHT HOLDER,2369,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,541909.0,NaN,NaN,NaN,9.55225,-80995.0,1.0,3.0,10.0,80995.0,218.081158
InvoiceDate,541909,NaN,NaN,NaN,2011-07-04 13:34:57.156386048,2010-12-01 08:26:00,2011-03-28 11:34:00,2011-07-19 17:17:00,2011-10-19 11:27:00,2011-12-09 12:50:00,NaN
UnitPrice,541909.0,NaN,NaN,NaN,4.611114,-11062.06,1.25,2.08,4.13,38970.0,96.759853
CustomerID,541909,4373,nan,135080,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country,541909,38,United Kingdom,495478,NaN,NaN,NaN,NaN,NaN,NaN,NaN


General observations:
- There are a total of 25,900 documents (invoices)
- The maximum number of rows in a document is 1,114
- There are 4,070 different products
- The most frequent product is "WHITE HANGING HEART T-LIGHT HOLDER" which is present in 2,369 rows
- There are 4,373 customers

Considerations on data quality:<br>
- *Quantity* and *UnitPrice* presents negative values. It happens in returns
- *Quantity* and *UnitPrice* also seem to present some outliers

In [7]:
# Check products cardinality
descriptionCount = ds['Description'].value_counts()
descriptionCount

Description
WHITE HANGING HEART T-LIGHT HOLDER     2369
REGENCY CAKESTAND 3 TIER               2200
JUMBO BAG RED RETROSPOT                2159
PARTY BUNTING                          1727
LUNCH BAG RED RETROSPOT                1638
                                       ... 
Missing                                   1
historic computer difference?....se       1
DUSTY PINK CHRISTMAS TREE 30CM            1
WRAP BLUE RUSSIAN FOLKART                 1
PINK BERTIE MOBILE PHONE CHARM            1
Name: count, Length: 4224, dtype: int64

Some "products" are not products, like the "missing?". Others were only sold once. These products could be deleted from the analysis.

In [8]:
# Check customers cardinality
ds['CustomerID'].value_counts()

CustomerID
nan        135080
17841.0      7983
14911.0      5903
14096.0      5128
12748.0      4642
            ...  
13270.0         1
17763.0         1
17291.0         1
15668.0         1
15562.0         1
Name: count, Length: 4373, dtype: int64

Looks like the "nan" customer is in fact a trasient or "missing" customer

## Data preparation

In [9]:
# Remove rows with out customer
ds = ds[ds['CustomerID']!='nan']

In [10]:
# Remove returns' rows
ds = ds[(ds['Quantity']>0) & (ds['UnitPrice']>0)]

In [11]:
def removeOutliers(df, colList, lowPercentile=0.05, highPercentile=0.95, verbose=False):
    """Remove rows with outliers below or above the predefined percentile thresholds

    Args:
        df (dataframe): dataframe object
        colList (string list): list with names of columns
        lowPercentile (float, optional): Rows with value below this threshold will be removed. Defaults to 0.05.
        highPercentile (float, optional): Rows with value above this threshold will be removed. Defaults to 0.95.
        verbose (bool, optional): Indication if it should give feedback on the percentiles. Defaults to False.

    Returns:
        dataframe: the dataframe with the rows removed.
    """
    # Identify percentiles
    quant_df = df[colList].quantile([lowPercentile, highPercentile])
    if verbose:
        print(quant_df)

    # Loop in each column
    for name in list(df[colList].columns):

        # Keep only rows that are inside the limits
        df = df[(df[name] >= quant_df.loc[lowPercentile, name]) & (df[name] <= quant_df.loc[highPercentile, name])]

    return df

In [12]:
# Remove outliers
cols = ['Quantity','UnitPrice']
ds = removeOutliers(ds, colList=cols , lowPercentile=0.00, highPercentile=0.99, verbose=True)

      Quantity  UnitPrice
0.00       1.0      0.001
0.99     120.0     14.950


In [13]:
# Remove not useful products
# Assuming the valid products usually have UPPERCASE descriptions

descWithLowercase = []

for description in descriptionCount.index:
    if any(c for c in description if c.islower()):
        descWithLowercase.append(description)

descWithLowercase

['nan',
 'Manual',
 'BAG 125g SWIRLY MARBLES',
 'BAG 250g SWIRLY MARBLES',
 '3 TRADITIONAl BISCUIT CUTTERS  SET',
 'POLYESTER FILLER PAD 40x40cm',
 'FRENCH BLUE METAL DOOR SIGN No',
 'check',
 'BAG 500g SWIRLY MARBLES',
 'POLYESTER FILLER PAD 45x45cm',
 'Next Day Carriage',
 'Discount',
 'damages',
 'damaged',
 'POLYESTER FILLER PAD 45x30cm',
 'Bank Charges',
 'NUMBER TILE VINTAGE FONT No ',
 'NUMBER TILE COTTAGE GARDEN No',
 'ESSENTIAL BALM 3.5g TIN IN ENVELOPE',
 'POLYESTER FILLER PAD 30CMx30CM',
 'found',
 'sold as set on dotcom',
 'CRUK Commission',
 'adjustment',
 'Damaged',
 'FOLK ART GREETING CARD,pack/12',
 'Dotcomgiftshop Gift Voucher £20.00',
 'Unsaleable, destroyed.',
 'thrown away',
 'amazon',
 'Found',
 'THE KING GIFT BAG 25x24x12cm',
 'Dotcomgiftshop Gift Voucher £10.00',
 'Dotcomgiftshop Gift Voucher £30.00',
 'Amazon',
 'POLYESTER FILLER PAD 65CMx65CM',
 'dotcom',
 'had been put aside',
 'wet damaged',
 'damages?',
 'ebay',
 'Dotcomgiftshop Gift Voucher £50.00',
 'smash

In [14]:
# Actually, some useful products have some lowercase letters
# We may use the difference between uppercase and lowercase letters

notUsefulProducts = []

for description in descriptionCount.index:
    uppercaseLetters = [c for c in description if c.isupper()]
    lowercaseLetters = [c for c in description if c.islower()]
    if len(lowercaseLetters) > len(uppercaseLetters):
        notUsefulProducts.append(description)

notUsefulProducts

['nan',
 'Manual',
 'check',
 'Next Day Carriage',
 'Discount',
 'damages',
 'damaged',
 'Bank Charges',
 'found',
 'sold as set on dotcom',
 'CRUK Commission',
 'adjustment',
 'Damaged',
 'Dotcomgiftshop Gift Voucher £20.00',
 'Unsaleable, destroyed.',
 'thrown away',
 'amazon',
 'Found',
 'Dotcomgiftshop Gift Voucher £10.00',
 'Dotcomgiftshop Gift Voucher £30.00',
 'Amazon',
 'dotcom',
 'had been put aside',
 'wet damaged',
 'damages?',
 'ebay',
 'Dotcomgiftshop Gift Voucher £50.00',
 'smashed',
 'Adjust bad debt',
 'test',
 'High Resolution Image',
 'Dotcomgiftshop Gift Voucher £40.00',
 'wet pallet',
 'missing',
 'mailout',
 'mixed up',
 '???missing',
 'rusty throw away',
 'wet rusty',
 'taig adjust',
 "Dr. Jam's Arouzer Stress Ball",
 'incorrect stock entry.',
 'wrongly coded 20713',
 'stock check',
 'sold as 1',
 'damages wax',
 '?missing',
 "Dad's Cab Electronic Meter",
 '*USB Office Mirror Ball',
 'Dotcom sales',
 'printing smudges/thrown away',
 'incorrectly credited C550456 s

In [15]:
# Remove not useful products
ds = ds[~ds['Description'].isin(notUsefulProducts)]

In [16]:
### Customer-item matrix
customerProductMatrix = ds.pivot_table(
                    index='CustomerID',
                    columns='Description',
                    values='Quantity',
                    aggfunc=lambda x: 1 if len(x)>0 else 0).fillna(0)

In [17]:
# See matrix for one specific customer
customerProductMatrix.head()

Description,4 PURPLE FLOCK DINNER CANDLES,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,I LOVE LONDON MINI RUCKSACK,NINE DRAWER OFFICE TIDY,OVAL WALL MIRROR DIAMANTE,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,...,ZINC PLANT POT HOLDER,ZINC STAR T-LIGHT HOLDER,ZINC SWEETHEART SOAP DISH,ZINC SWEETHEART WIRE LETTER RACK,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC WILLIE WINKIE CANDLE STICK,ZINC WIRE KITCHEN ORGANISER,ZINC WIRE SWEETHEART LETTER TRAY
CustomerID,,,,,,,,,,,,,,,,,,,,,
12347.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12348.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12349.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12350.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12352.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [18]:
# Shape of the matrix
customerProductMatrix.shape

(4290, 3829)

<br><br><br><br><br>
## Customer based recommendations
### Based on similar customers' purchases

In [19]:
# Create a customer to customer similarity matrix
# Info on the "cosine" implementation https://docs.scipy.org/doc/scipy/reference/reference/generated/scipy.spatial.distance.cosine.html#scipy.spatial.distance.cosine
customer_customer_sim_matrix = pd.DataFrame(
    pairwise_distances(customerProductMatrix,metric='cosine'),
    columns = customerProductMatrix.index,
    index = customerProductMatrix.index
)
customer_customer_sim_matrix = customer_customer_sim_matrix.apply(lambda x: 1-x, axis=1) # Transform dissimilarity to similarity
customer_customer_sim_matrix

CustomerID,12347.0,12348.0,12349.0,12350.0,12352.0,12353.0,12354.0,12355.0,12356.0,12357.0,...,18273.0,18274.0,18276.0,18277.0,18278.0,18280.0,18281.0,18282.0,18283.0,18287.0
CustomerID,,,,,,,,,,,,,,,,,,,,,
12347.0,1.000000,0.051131,0.047338,0.049507,0.039344,0.0,0.013351,0.109847,0.096116,0.060789,...,0.0,0.029854,0.026463,0.000000,0.033005,0.062622,0.000000,0.114332,0.102439,0.000000
12348.0,0.051131,1.000000,0.000000,0.000000,0.000000,0.0,0.034816,0.143223,0.143223,0.000000,...,0.0,0.077850,0.138013,0.000000,0.000000,0.000000,0.000000,0.000000,0.125708,0.100844
12349.0,0.047338,0.000000,1.000000,0.029881,0.110818,0.0,0.032233,0.033150,0.132599,0.157243,...,0.0,0.108112,0.000000,0.000000,0.039841,0.000000,0.000000,0.069007,0.109109,0.015561
12350.0,0.049507,0.000000,0.029881,1.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.021926,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.045644,0.000000
12352.0,0.039344,0.000000,0.110818,0.000000,1.000000,0.0,0.107160,0.036736,0.073472,0.058085,...,0.0,0.159745,0.000000,0.000000,0.000000,0.000000,0.000000,0.038236,0.072548,0.034488
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18280.0,0.062622,0.000000,0.000000,0.000000,0.000000,0.0,0.042640,0.000000,0.000000,0.055470,...,0.0,0.000000,0.000000,0.000000,0.105409,1.000000,0.141421,0.000000,0.000000,0.000000
18281.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.078446,...,0.0,0.000000,0.000000,0.000000,0.000000,0.141421,1.000000,0.000000,0.054433,0.000000
18282.0,0.114332,0.000000,0.069007,0.000000,0.038236,0.0,0.000000,0.160128,0.080064,0.025318,...,0.0,0.174078,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.017568,0.000000


In [20]:
# Check the matrix results
customer_customer_sim_matrix.head()

CustomerID,12347.0,12348.0,12349.0,12350.0,12352.0,12353.0,12354.0,12355.0,12356.0,12357.0,...,18273.0,18274.0,18276.0,18277.0,18278.0,18280.0,18281.0,18282.0,18283.0,18287.0
CustomerID,,,,,,,,,,,,,,,,,,,,,
12347.0,1.000000,0.051131,0.047338,0.049507,0.039344,0.0,0.013351,0.109847,0.096116,0.060789,...,0.0,0.029854,0.026463,0.0,0.033005,0.062622,0.0,0.114332,0.102439,0.000000
12348.0,0.051131,1.000000,0.000000,0.000000,0.000000,0.0,0.034816,0.143223,0.143223,0.000000,...,0.0,0.077850,0.138013,0.0,0.000000,0.000000,0.0,0.000000,0.125708,0.100844
12349.0,0.047338,0.000000,1.000000,0.029881,0.110818,0.0,0.032233,0.033150,0.132599,0.157243,...,0.0,0.108112,0.000000,0.0,0.039841,0.000000,0.0,0.069007,0.109109,0.015561
12350.0,0.049507,0.000000,0.029881,1.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.021926,...,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.045644,0.000000
12352.0,0.039344,0.000000,0.110818,0.000000,1.000000,0.0,0.107160,0.036736,0.073472,0.058085,...,0.0,0.159745,0.000000,0.0,0.000000,0.000000,0.0,0.038236,0.072548,0.034488


In [21]:
# Similar customers
customer_customer_sim_matrix.loc['12350.0'].sort_values(ascending=False)

CustomerID
12350.0    1.000000
16426.0    0.172107
17486.0    0.153093
16846.0    0.150756
12652.0    0.145865
             ...   
14658.0    0.000000
14659.0    0.000000
14660.0    0.000000
14661.0    0.000000
18287.0    0.000000
Name: 12350.0, Length: 4290, dtype: float64

In [22]:
# Check products bought by A
nonZeroProducts = (customerProductMatrix.loc['12350.0'] !=0)
products_bought_by_A = set(customerProductMatrix.columns[nonZeroProducts])
products_bought_by_A

{'4 TRADITIONAL SPINNING TOPS',
 'BATHROOM METAL SIGN ',
 'BLUE POLKADOT LUGGAGE TAG ',
 'BLUE POLKADOT PASSPORT COVER',
 'CHOCOLATE CALCULATOR',
 'CHOCOLATE THIS WAY METAL SIGN',
 'METAL SIGN NEIGHBOURHOOD WITCH ',
 'PINK/PURPLE RETRO RADIO',
 'PLASTERS IN TIN SPACEBOY',
 'PLASTERS IN TIN VINTAGE PAISLEY ',
 'RED  HARMONICA IN BOX ',
 'RETRO MOD TRAY',
 'RETRO PLASTIC ELEPHANT TRAY',
 'TEA BAG PLATE RED RETROSPOT',
 'UNION JACK FLAG LUGGAGE TAG',
 'UNION JACK FLAG PASSPORT COVER '}

In [23]:
# Check products bought by B
nonZeroProducts = (customerProductMatrix.loc['16426.0'] !=0)
products_bought_by_B = set(customerProductMatrix.columns[nonZeroProducts])
products_bought_by_B

{'12 PENCIL SMALL TUBE WOODLAND',
 '3 HOOK PHOTO SHELF ANTIQUE WHITE',
 '3 STRIPEY MICE FELTCRAFT',
 '36 PENCILS TUBE RED RETROSPOT',
 '36 PENCILS TUBE SKULLS',
 '4 TRADITIONAL SPINNING TOPS',
 'AIRLINE BAG VINTAGE JET SET BROWN',
 'AIRLINE BAG VINTAGE JET SET RED',
 'AIRLINE BAG VINTAGE JET SET WHITE',
 'AIRLINE BAG VINTAGE TOKYO 78',
 'AIRLINE BAG VINTAGE WORLD CHAMPION ',
 'AIRLINE LOUNGE,METAL SIGN',
 'ALARM CLOCK BAKELIKE GREEN',
 'ALARM CLOCK BAKELIKE IVORY',
 'ALARM CLOCK BAKELIKE RED ',
 'ANTIQUE CREAM CUTLERY SHELF ',
 'BEWARE OF THE CAT METAL SIGN ',
 'BLACK/BLUE POLKADOT UMBRELLA',
 'BLUE CHARLIE+LOLA PERSONAL DOORSIGN',
 'BLUE POLKADOT LUGGAGE TAG ',
 'BLUE POLKADOT PASSPORT COVER',
 'BULL DOG BOTTLE TOP WALL CLOCK',
 'CHOCOLATE CALCULATOR',
 'CHOCOLATE THIS WAY METAL SIGN',
 'CHRISTMAS MUSICAL ZINC STAR',
 'CHRISTMAS MUSICAL ZINC TREE',
 'CHRISTMAS RETROSPOT STAR WOOD',
 'CHRISTMAS RETROSPOT TREE WOOD',
 'COLUMBIAN CANDLE ROUND',
 'COOK WITH WINE METAL SIGN ',
 'CREAM CUPI

In [24]:
# Product to recommend to B
product_to_recommend_to_B = products_bought_by_A - products_bought_by_B
product_to_recommend_to_B

{'BATHROOM METAL SIGN ',
 'PINK/PURPLE RETRO RADIO',
 'RED  HARMONICA IN BOX ',
 'RETRO MOD TRAY',
 'RETRO PLASTIC ELEPHANT TRAY',
 'TEA BAG PLATE RED RETROSPOT'}

### Based on products similarity

In [25]:
# Similiar products matrix
product_product_sim_matrix = pd.DataFrame(
    pairwise_distances(customerProductMatrix.T,metric='cosine'),
    columns = customerProductMatrix.columns,
    index = customerProductMatrix.columns
    )
product_product_sim_matrix = product_product_sim_matrix.apply(lambda x: 1-x, axis=1) # Transform dissimilarity to similarity
product_product_sim_matrix

Description,4 PURPLE FLOCK DINNER CANDLES,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,I LOVE LONDON MINI RUCKSACK,NINE DRAWER OFFICE TIDY,OVAL WALL MIRROR DIAMANTE,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,...,ZINC PLANT POT HOLDER,ZINC STAR T-LIGHT HOLDER,ZINC SWEETHEART SOAP DISH,ZINC SWEETHEART WIRE LETTER RACK,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC WILLIE WINKIE CANDLE STICK,ZINC WIRE KITCHEN ORGANISER,ZINC WIRE SWEETHEART LETTER TRAY
Description,,,,,,,,,,,,,,,,,,,,,
4 PURPLE FLOCK DINNER CANDLES,1.000000,0.000000,0.018051,0.023583,0.000000,0.000000,0.028050,0.000000,0.031505,0.038292,...,0.000000,0.000000,0.042333,0.043885,0.032001,0.000000,0.027154,0.061379,0.000000,0.042333
50'S CHRISTMAS GIFT BAG LARGE,0.000000,1.000000,0.058849,0.038443,0.000000,0.036244,0.060964,0.325300,0.025678,0.000000,...,0.097590,0.000000,0.046004,0.047690,0.034776,0.000000,0.095902,0.033351,0.000000,0.023002
DOLLY GIRL BEAKER,0.018051,0.058849,1.000000,0.145165,0.100504,0.037326,0.047088,0.067003,0.061703,0.085710,...,0.100504,0.000000,0.023689,0.049114,0.089536,0.000000,0.091168,0.034346,0.029013,0.071067
I LOVE LONDON MINI BACKPACK,0.023583,0.038443,0.145165,1.000000,0.131306,0.048766,0.041013,0.043769,0.126680,0.055989,...,0.000000,0.000000,0.061898,0.048125,0.035093,0.000000,0.029778,0.056091,0.000000,0.061898
I LOVE LONDON MINI RUCKSACK,0.000000,0.000000,0.100504,0.131306,1.000000,0.000000,0.000000,0.000000,0.000000,0.213201,...,0.000000,0.000000,0.000000,0.000000,0.089087,0.000000,0.000000,0.085436,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZINC T-LIGHT HOLDER STARS LARGE,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.106904,0.000000,0.000000,0.000000
ZINC T-LIGHT HOLDER STARS SMALL,0.027154,0.095902,0.091168,0.029778,0.000000,0.042112,0.082639,0.050395,0.053039,0.016116,...,0.000000,0.075593,0.071270,0.138527,0.552217,0.106904,1.000000,0.122708,0.043644,0.071270
ZINC WILLIE WINKIE CANDLE STICK,0.061379,0.033351,0.034346,0.056091,0.085436,0.031730,0.053371,0.047464,0.014986,0.036430,...,0.000000,0.000000,0.040275,0.083501,0.098946,0.000000,0.122708,1.000000,0.024663,0.040275


In [26]:
# Top 10 products similar to ' 4 PURPLE FLOCK DINNER CANDLES'
top_10_similar_items = list(
    product_product_sim_matrix
        .loc[' 4 PURPLE FLOCK DINNER CANDLES']
        .sort_values(ascending=False)
        .iloc[1:11]         # 1 to 11 instead of 0 to 10 because the first is the product itself
    .index
)
top_10_similar_items

['4 PINK DINNER CANDLE SILVER FLOCK',
 '4 BLUE DINNER CANDLES SILVER FLOCK',
 '4 IVORY DINNER CANDLES SILVER FLOCK',
 '4 PINK FLOCK CHRISTMAS BALLS',
 'DOLPHIN WINDMILL',
 'FRYING PAN RED POLKADOT ',
 'GLASS CAKE COVER AND PLATE',
 'SET/3 FLORAL GARDEN TOOLS IN BAG',
 'HEN HOUSE W CHICK IN NEST',
 'POLYESTER FILLER PAD 60x40cm']

### **EXERCISE 1**
Check how results change:

- Using different measures (check the different measures in https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise_distances.html)
- With other customers
- With other products